# Can we predict 12-month loan default?

**Question:** using only information available at application time, can we predict `defaulted` (1 = default within 12 months of funding)?

**Plan:** check data quality → EDA → hold out a test set → compare a naive baseline, logistic regression and gradient boosting using cross-validation → evaluate once on the test set → look at what drives predictions and whether the model is useful for ranking risk.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width',160)
SEED=42
df=pd.read_csv('data.csv'); cols=pd.read_csv('columns.csv')
print(df.shape); cols

## 1. Data quality

In [ ]:
print('missing values:\n', df.isna().sum().to_string())
print('duplicate rows:', df.duplicated().sum(), '| duplicate ids:', df.application_id.duplicated().sum())
df.describe().T

In [ ]:
# Sanity checks on plausible ranges
print('employment_years > age-16:', (df.employment_years > df.age-16).sum())
print('zero employment share:', (df.employment_years==0).mean().round(3))
print('debt_ratio > 1:', (df.debt_ratio>1).sum())
print(df.region.value_counts().to_string())

## 2. Target and univariate relationships

In [ ]:
rate=df.defaulted.mean()
print(f'Default rate: {rate:.3%}  ({df.defaulted.sum()} of {len(df)})')
num=['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
fig,ax=plt.subplots(2,3,figsize=(14,7))
for a,c in zip(ax.ravel(),num):
    q=pd.qcut(df[c],10,duplicates='drop')
    g=df.groupby(q,observed=True).defaulted.mean()
    a.plot(range(len(g)),g.values,'o-'); a.axhline(rate,color='grey',ls='--')
    a.set_title(f'Default rate by {c} decile'); a.set_xlabel('decile (low→high)'); a.set_ylim(0,None)
plt.tight_layout(); plt.show()
print(df.groupby('region').defaulted.agg(['mean','size']))

In [ ]:
print('Correlation with defaulted:'); print(df[num+['defaulted']].corr()['defaulted'].drop('defaulted').round(3).to_string())
print('\napplication_id vs defaulted (should be ~0, it is just an identifier):', round(df.application_id.corr(df.defaulted),3))

`application_id` is an identifier and is **excluded** from modelling. All other columns are recorded at application time, so there is no obvious target leakage.

## 3. Train/test split
The data has no date column, so a stratified random split is used. 20% is held out and not touched until the final evaluation.

In [ ]:
regions=sorted(df.region.unique())
feats=['region']+num
X,y=df[feats].copy(),df.defaulted
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
# integer region codes for the tree model
def to_codes(d):
    d=d.copy(); d['region']=pd.Categorical(d.region,categories=regions).codes; return d
Xtr_gb,Xte_gb=to_codes(Xtr),to_codes(Xte)
print(len(Xtr),len(Xte),ytr.mean().round(4),yte.mean().round(4))

## 4. Models (5-fold stratified CV on the training set)
Metrics: ROC AUC (ranking quality), PR AUC / average precision (relevant because defaults are rare; a random model scores ≈ the base rate) and Brier score (probability quality).

In [ ]:
pre=ColumnTransformer([('num',StandardScaler(),num),('cat',OneHotEncoder(drop='first'),['region'])])
models={
 'baseline (prior)':(DummyClassifier(strategy='prior'),False),
 'logistic regression':(make_pipeline(pre,LogisticRegression(max_iter=1000)),False),
 'gradient boosting':(HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,l2_regularization=1.0,
        categorical_features=[0],random_state=SEED),True),
}
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
rows=[]
for n,(m,gb_in) in models.items():
    Xi=Xtr_gb if gb_in else Xtr
    p=cross_val_predict(m,Xi,ytr,cv=cv,method='predict_proba')[:,1]
    s=cross_val_score(m,Xi,ytr,cv=cv,scoring='roc_auc')
    rows.append((n,roc_auc_score(ytr,p),s.std(),average_precision_score(ytr,p),brier_score_loss(ytr,p)))
pd.DataFrame(rows,columns=['model','ROC AUC (pooled OOF)','AUC fold sd','PR AUC','Brier']).set_index('model').round(4)

## 5. Final evaluation on the held-out test set

In [ ]:
lr=models['logistic regression'][0].fit(Xtr,ytr)
gb=models['gradient boosting'][0].fit(Xtr_gb,ytr)
final={'baseline (prior)':np.full(len(yte),ytr.mean()),
       'logistic regression':lr.predict_proba(Xte)[:,1],
       'gradient boosting':gb.predict_proba(Xte_gb)[:,1]}
rng=np.random.default_rng(SEED); yv=yte.values
def boot(p,B=1000):
    v=[]
    for _ in range(B):
        i=rng.integers(0,len(yv),len(yv))
        if len(set(yv[i]))==2: v.append(roc_auc_score(yv[i],p[i]))
    return np.percentile(v,[2.5,97.5])
rows=[]
for n,p in final.items():
    lo,hi=boot(p) if n!='baseline (prior)' else (np.nan,np.nan)
    rows.append((n,roc_auc_score(yte,p),lo,hi,average_precision_score(yte,p),brier_score_loss(yte,p)))
pd.DataFrame(rows,columns=['model','ROC AUC','AUC 2.5%','AUC 97.5%','PR AUC','Brier']).set_index('model').round(4)

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4.5))
for n in ['logistic regression','gradient boosting']:
    f,t,_=roc_curve(yte,final[n]); ax[0].plot(f,t,label=f'{n} ({roc_auc_score(yte,final[n]):.3f})')
    pr,rc,_=precision_recall_curve(yte,final[n]); ax[1].plot(rc,pr,label=n)
    fp,mp=calibration_curve(yte,final[n],n_bins=8,strategy='quantile'); ax[2].plot(mp,fp,'o-',label=n)
ax[0].plot([0,1],[0,1],'k--'); ax[0].set(title='ROC (test)',xlabel='FPR',ylabel='TPR')
ax[1].axhline(yte.mean(),color='k',ls='--'); ax[1].set(title='Precision-recall (test)',xlabel='recall',ylabel='precision')
m=max(ax[2].get_xlim()[1],ax[2].get_ylim()[1]); ax[2].plot([0,m],[0,m],'k--'); ax[2].set(title='Calibration (test)',xlabel='mean predicted',ylabel='observed rate')
for a in ax: a.legend()
plt.tight_layout(); plt.show()

## 6. What drives the predictions?

In [ ]:
names=num+[f'region_{r}' for r in regions[1:]]
coef=pd.Series(lr[-1].coef_[0],index=names)
print('Logistic regression coefficients (numerics standardised; log-odds per 1 SD):'); print(coef.round(3).to_string())
pi=permutation_importance(gb,Xte_gb,yte,scoring='roc_auc',n_repeats=20,random_state=SEED)
print('\nGradient boosting permutation importance on test set (drop in AUC):')
pd.DataFrame({'AUC drop':pi.importances_mean,'sd':pi.importances_std},index=feats).sort_values('AUC drop',ascending=False).round(4)

## 7. Practical usefulness: risk-ranked lift

In [ ]:
best='gradient boosting' if roc_auc_score(yte,final['gradient boosting'])>=roc_auc_score(yte,final['logistic regression']) else 'logistic regression'
t=pd.DataFrame({'y':yv,'p':final[best]}); t['decile']=pd.qcut(t.p.rank(method='first'),10,labels=range(1,11))
g=t.groupby('decile',observed=True).agg(n=('y','size'),default_rate=('y','mean'),mean_pred=('p','mean'))
g['lift']=g.default_rate/yv.mean()
print('model:',best); print(g.round(3).to_string())
top=t.nlargest(int(len(t)*.2),'p'); print(f'\nTop 20% riskiest capture {top.y.sum()/t.y.sum():.1%} of all defaults (random would be 20%).')

## 8. Conclusions (from a run with `SEED=42`)

**Yes, default is moderately predictable, but far from perfectly.**

- Base rate is ~10.5% defaults (837 / 8,000), so accuracy is a misleading metric; a model that always says "no default" is 89.5% accurate.
- Held-out test ROC AUC ≈ **0.775** for logistic regression (bootstrap 95% CI ≈ 0.74–0.81) vs 0.50 for the baseline. PR AUC ≈ 0.35 vs a 0.10 baseline.
- Gradient boosting (≈ 0.76) does **not** beat logistic regression; the difference is well within the CV noise (fold sd ≈ 0.02). The signal looks mostly additive/monotone, so the simpler, interpretable model is preferred.
- Main drivers: **higher `debt_ratio`** (strongest), **lower `income_k`**, **shorter `employment_years`** and **larger `loan_amount_k`** raise risk. `credit_lines`, `age` and `region` add essentially nothing.
- Ranking is useful: the riskiest decile defaults at ~36% (≈3.5× base rate) and the safest 20% at <1.5%; the top 20% of scores capture ~53% of all defaults.
- Probabilities are reasonably calibrated (Brier 0.081 vs 0.094 baseline).

**Caveats**
- 643 rows have `employment_years > age − 16`, which is implausible and suggests data-entry issues; worth checking with the data owner (it was left as is).
- Random split, no time dimension: performance on future cohorts (economic shifts, policy changes) may be lower.
- Only ~170 defaults in the test set, so the metrics carry noticeable uncertainty (see CI).
- Predictive, not causal, and no fairness review has been done; `age` and `region` should be reviewed before any real use.
- Choosing an approval threshold needs the actual cost of a default vs. a lost good loan, which we don't have.